In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("Day12_Used_Car_Preprocessing_Dataset.csv")

# Basic EDA & Inspection (as shown in lecture)
print("Dataset Shape:", df.shape)
print("\n--- Info ---")
print(df.info())
print("\n--- Missing Values ---")
print(df.isnull().sum())

Dataset Shape: (320, 15)

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car_ID              320 non-null    object 
 1   Brand               320 non-null    object 
 2   Year                320 non-null    int64  
 3   Mileage_Km          320 non-null    int64  
 4   Engine_CC           320 non-null    int64  
 5   Power_BHP           320 non-null    float64
 6   Fuel_Type           320 non-null    object 
 7   Transmission        320 non-null    object 
 8   City                320 non-null    object 
 9   Seller_Type         320 non-null    object 
 10  Condition           320 non-null    object 
 11  Previous_Owners     320 non-null    int64  
 12  Accidents_Reported  320 non-null    int64  
 13  Service_Score       320 non-null    int64  
 14  Resale_Price_Lakh   320 non-null    float64
dtypes: float64(2), int

In [6]:
# Outlier detection and treatment using IQR method on target/price column
Q1 = df["Resale_Price_Lakh"].quantile(0.25)
Q3 = df["Resale_Price_Lakh"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

# Clipping outliers (Winsorization) as discussed in the lecture
df["Resale_Price_Lakh"] = df["Resale_Price_Lakh"].clip(lower=lower_limit, upper=upper_limit)
print("Outliers treated via IQR clipping successfully!")

Outliers treated via IQR clipping successfully!


In [7]:
# Ordinal mapping for Condition column
condition_map = {"Poor": 1, "Fair": 2, "Good": 3, "Very Good": 4, "Excellent": 5}
df["Encoded_Condition"] = df["Condition"].map(condition_map)

# One-hot encoding for nominal features with drop_first=True
df_encoded = pd.get_dummies(df, columns=["Fuel_Type", "Transmission", "City", "Seller_Type"], drop_first=True)
print("One-hot encoding applied. Total columns:", len(df_encoded.columns))

One-hot encoding applied. Total columns: 27


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define feature columns
feature_cols = ['Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Previous_Owners',
                'Accidents_Reported', 'Service_Score', 'Encoded_Condition']
for col in df_encoded.columns:
    if col.startswith(('Fuel_Type_', 'Transmission_', 'City_', 'Seller_Type_')):
        feature_cols.append(col)

X = df_encoded[feature_cols]
y = df_encoded["Resale_Price_Lakh"]

# Train-Test Split (80% train, 20% test with random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling using StandardScaler (Fit ONLY on training data to prevent data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Verify processed train dataset structure
df_processed = pd.DataFrame(X_train_scaled, columns=feature_cols)
df_processed["Resale_Price_Lakh"] = y_train.values

print("Processed Train Shape:", df_processed.shape)
print("Remaining Nulls:", df_processed.isnull().sum().sum())

# Export the final processed dataset
df_processed.to_csv("Preprocessed_Used_Car_Dataset.csv", index=False)
print("Preprocessed dataset successfully exported as 'Preprocessed_Used_Car_Dataset.csv'!")

Processed Train Shape: (256, 24)
Remaining Nulls: 0
Preprocessed dataset successfully exported as 'Preprocessed_Used_Car_Dataset.csv'!
